# 1.5) NumPy

Numpy is the foundation of scientific python: a dense, typed n-dimensional array (`ndarray`) and a large set of operations that act on whole arrays at once, without python-level loops. This subchapter works with one example throughout — a small two-dimensional temperature field on a latitude–longitude grid — to cover array creation, dtype and shape, indexing and masking, vectorised math and broadcasting, reductions, and the handling of missing data. It closes with a generated-code bug that silently truncates results because of a dtype mistake.

:::{admonition} Learning objectives
:class: tip
- Create arrays and inspect their `shape`, `ndim`, and `dtype`, and understand `float32` vs `float64` precision.
- Index, slice, and select elements with boolean masks.
- Replace element-wise loops with vectorised math and broadcasting.
- Reduce along chosen axes, and reshape and stack arrays.
- Choose values with `np.where`, handle missing data with NaN-aware operations, and fill gaps with `np.interp`.
- Generate reproducible random numbers with a seeded `default_rng`.
:::

## Creating arrays; shape, ndim, and dtype

An array is created from data or from a constructor. Every array carries a `shape` (size along each axis), an `ndim` (number of axes), and a `dtype` (the element type, which fixes both behaviour and memory use). A seeded `default_rng` gives reproducible random data.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)          # seeded generator: reproducible output

# a 2D field: 4 latitudes (rows) x 6 longitudes (cols), daily mean temp (°C)
temp_celsius = rng.normal(loc=3.0, scale=4.0, size=(4, 6))

print(temp_celsius.shape, temp_celsius.ndim, temp_celsius.dtype)
print(temp_celsius.round(2))

# common constructors
print(np.zeros(3), np.arange(0, 10, 2), np.linspace(0.0, 1.0, 5))

(4, 6) 2 float64
[[ 3.5   2.47  5.56  3.42  0.86  4.45]
 [ 8.22  6.79  0.19 -2.06  0.51  3.17]
 [-6.3   2.12 -1.98  0.07  0.82  1.73]
 [ 4.65  7.17  2.49  8.47  0.34  4.41]]
[0. 0. 0.] [0 2 4 6 8] [0.   0.25 0.5  0.75 1.  ]


## dtype and precision

float64 (double precision) is the default; float32 halves memory at the cost of precision. The choice matters for large fields and for the smallest differences a computation can resolve.

In [2]:
temp32 = temp_celsius.astype(np.float32)
print(temp_celsius.dtype, temp_celsius.nbytes, "bytes")
print(temp32.dtype, temp32.nbytes, "bytes")

# float32 is coarser: the rounding error in 0.1 + 0.2 is invisible at float32
print(np.float64(0.1) + np.float64(0.2))   # 0.30000000000000004
print(np.float32(0.1) + np.float32(0.2))   # 0.3 (coarser resolution hides it)

float64 192 bytes
float32 96 bytes
0.30000000000000004
0.3


## Indexing, slicing, and boolean masking

Indexing uses `[row, col]`; slicing selects sub-blocks; negative indices count from the end. A boolean *mask* is a same-shaped array of True/False that selects the matching elements.

In [3]:
print(temp_celsius[0, 0].round(2))      # one element
print(temp_celsius[0].round(2))         # first row, all longitudes
print(temp_celsius[:, -1].round(2))     # last column, all latitudes
print(temp_celsius[1:3, 2:4].round(2))  # a 2x2 sub-block

# a boolean mask and what it selects
freezing = temp_celsius < 0.0
print("n freezing cells:", int(freezing.sum()))            # True counts as 1
print("freezing values:", temp_celsius[freezing].round(2)) # 1D of matches

3.5
[3.5  2.47 5.56 3.42 0.86 4.45]
[4.45 3.17 1.73 4.41]
[[ 0.19 -2.06]
 [-1.98  0.07]]
n freezing cells: 3
freezing values: [-2.06 -6.3  -1.98]


:::{admonition} Computational-thinking fundamental: think in arrays, not loops
:class: important
The central idea of numpy is *vectorisation*: express a computation as an operation on whole arrays rather than a loop over elements. `field + 273.15` converts every value at once. Vectorised code is shorter, far faster (the loop runs in compiled C), and closer to the mathematical statement of the problem. When you find yourself writing a python `for` loop over array elements, look for the array operation that replaces it.
:::

## Vectorised math and broadcasting

A single expression applies element-wise across an array. *Broadcasting* lets arrays of different but compatible shapes combine: a length-4 column vector can be stretched across all 6 longitudes.

In [4]:
# vectorised: no python loop
temp_kelvin = temp_celsius + 273.15
print(temp_kelvin.round(2)[0])           # first row, in kelvin

# broadcasting: a (4,1) column stretches across the 6 longitudes
lat_gradient_celsius = np.array([0.0, -1.5, -3.0, -4.5])   # colder toward the north
adjusted = temp_celsius + lat_gradient_celsius[:, None]    # (4,1) + (4,6) -> (4,6)
print(adjusted.shape)
print(adjusted.round(2))

[276.65 275.62 278.71 276.57 274.01 277.6 ]
(4, 6)
[[ 3.5   2.47  5.56  3.42  0.86  4.45]
 [ 6.72  5.29 -1.31 -3.56 -0.99  1.67]
 [-9.3  -0.88 -4.98 -2.93 -2.18 -1.27]
 [ 0.15  2.67 -2.01  3.97 -4.16 -0.09]]


## Reductions, reshape, and stacking

A reduction collapses an axis: `axis=0` aggregates over latitudes (one result per longitude), `axis=1` over longitudes. `reshape` reorganises the same data; stacking combines arrays.

In [5]:
print("overall mean:", temp_celsius.mean().round(2))
print("mean per longitude (over lat):", temp_celsius.mean(axis=0).round(2))
print("mean per latitude (over lon):", temp_celsius.mean(axis=1).round(2))

# reshape: same 24 values, new shape; -1 infers the missing length
flat = temp_celsius.reshape(-1)
print(flat.shape)

# stacking: combine arrays along a new row axis
col_means = temp_celsius.mean(axis=0)
index_row = np.arange(6, dtype=float)
print(np.vstack([index_row, col_means]).shape)   # (2, 6)

overall mean: 2.54
mean per longitude (over lat): [2.52 4.64 1.56 2.47 0.63 3.44]
mean per latitude (over lon): [ 3.38  2.8  -0.59  4.59]
(24,)
(2, 6)


:::{admonition} Quick exercise: warmest latitude
:class: note
Compute the mean temperature of each latitude (each row), then use `argmax` to find the index of the warmest latitude.
:::

:::{admonition} Solution
:class: note dropdown
```python
row_means = temp_celsius.mean(axis=1)
print(row_means.round(2))
print(int(row_means.argmax()))
```
:::

## Choosing, missing data, and interpolation

`np.where` picks element-wise between two options. Missing data is represented by `NaN`; NaN-aware reductions (`np.nanmean`, …) skip it, while ordinary reductions propagate it. `np.interp` fills a 1D gap by linear interpolation.

In [6]:
# np.where(condition, a, b): element-wise choice
category = np.where(temp_celsius < 0.0, "freezing", "above")
print(category)

# missing data as NaN; nan-aware vs ordinary reduction
temp_with_gaps = temp_celsius.copy()
temp_with_gaps[0, 0] = np.nan
print("nanmean (skips gaps):", np.nanmean(temp_with_gaps).round(2))
print("plain mean is contaminated:", np.mean(temp_with_gaps).round(2))  # nan

# np.interp: fill a 1D gap by linear interpolation against an index
profile = temp_celsius[:, 0].copy()      # the first column, 4 latitudes
x = np.arange(profile.size)
known = np.array([0, 1, 3])              # pretend index 2 is missing
filled = np.interp(x, known, profile[known])
print(profile.round(2))
print(filled.round(2))                   # index 2 interpolated from its neighbours

[['above' 'above' 'above' 'above' 'above' 'above']
 ['above' 'above' 'above' 'freezing' 'above' 'above']
 ['freezing' 'above' 'freezing' 'above' 'above' 'above']
 ['above' 'above' 'above' 'above' 'above' 'above']]
nanmean (skips gaps): 2.5
plain mean is contaminated: nan
[ 3.5   8.22 -6.3   4.65]
[3.5  8.22 6.43 4.65]


## When generated code lies: a silent dtype truncation

ai assistants often preallocate an output array with `np.zeros_like`, which copies the *input's* dtype. If the input is integer, float results are silently truncated on assignment. Here a function computes anomalies (value minus the field mean) for an integer precipitation field.

In [7]:
def to_anomaly(field):
    # subtract the mean into a preallocated array (as an assistant returned it)
    result = np.zeros_like(field)        # inherits field's dtype!
    result[:] = field - field.mean()
    return result

precip_mm = np.array([[0, 2, 5], [1, 0, 8], [3, 4, 2]])   # integer mm
print("input dtype:", precip_mm.dtype)
print(to_anomaly(precip_mm))

input dtype: int64
[[-2  0  2]
 [-1 -2  5]
 [ 0  1  0]]


:::{admonition} Diagnosis: the output inherited an integer dtype
:class: warning
The anomalies should be fractional, but every value is a whole number. `np.zeros_like(field)` produced an *integer* array because `field` is integer, and assigning float anomalies into it truncates each toward zero. The error is silent — no exception, just wrong numbers. The fix is to let numpy choose the result type, or to request `dtype=float` explicitly.
:::

In [8]:
def to_anomaly(field):
    # let numpy promote to float; no wrong-dtype preallocation
    return field - field.mean()

print(to_anomaly(precip_mm).round(2))
print("output dtype:", to_anomaly(precip_mm).dtype)

[[-2.78 -0.78  2.22]
 [-1.78 -2.78  5.22]
 [ 0.22  1.22 -0.78]]
output dtype: float64


:::{admonition} Going deeper: memory layout, views, and strides
:class: seealso dropdown
An array is a flat block of memory plus a `shape` and `strides` (the byte step along each axis). Slicing returns a *view* that shares memory with the original, so writing through it mutates the source; use `.copy()` for an independent array. C-order (row-major, the default) and Fortran-order (column-major) change which axis is contiguous and therefore which traversals are fastest.

```python
a = np.arange(12).reshape(3, 4)
print(a.strides)          # bytes to step along (rows, cols)
b = a[:, 1]               # a view, not a copy
b[0] = 999                # this also changes a[0, 1]
```
:::

:::{admonition} Going deeper: vectorisation vs loops
:class: seealso dropdown
Vectorised array operations run in compiled code and are typically one to two orders of magnitude faster than an equivalent python loop. You can measure it in a notebook:

```python
big = np.random.default_rng(0).random(1_000_000)
%timeit big + 1.0                       # vectorised
%timeit [x + 1.0 for x in big]          # python loop, much slower
```

The gap widens with array size. Reach for the array expression first; drop to a loop only when no vectorised form exists.
:::

:::{admonition} Going deeper: basic linear algebra
:class: seealso dropdown
numpy covers the everyday linear algebra a model needs.

```python
A = np.array([[2.0, 1.0], [1.0, 3.0]])
b = np.array([1.0, 2.0])
print(A @ b)                 # matrix-vector product (also np.matmul)
print(np.linalg.solve(A, b)) # solve A x = b without inverting A
```

Prefer `np.linalg.solve` over forming `np.linalg.inv(A) @ b`: it is more accurate and faster.
:::

:::{admonition} Going deeper: bigger-than-memory arrays with dask
:class: seealso dropdown
When a field is too large for memory, `dask.array` exposes the same numpy interface over *chunks*, building a task graph that runs only when you call `.compute()`.

```python
import dask.array as da
x = da.from_array(np.arange(1_000_000), chunks=100_000)
result = (x + 1).mean()      # lazy: nothing computed yet
print(result.compute())      # runs the graph, chunk by chunk
```

This previews the lazy, chunked model that xarray uses for climate-scale datasets in later subchapters.
:::

:::{admonition} Takeaways
:class: danger
- An array carries shape, ndim, and dtype; the dtype fixes both precision and memory (float64 default, float32 half the size and coarser).
- Index and slice with `[row, col]`; select with boolean masks; views share memory, so copy when you need independence.
- Vectorise: write array expressions instead of element loops, and use broadcasting to combine compatible shapes.
- Reduce along an explicit `axis`; reshape and stack to reorganise data.
- Use `np.where` to choose element-wise, NaN-aware ops for missing data, and `np.interp` to fill 1D gaps.
- Watch dtype on preallocation: `np.zeros_like(int_array)` truncates float results silently — let numpy promote to float.
:::

## Resources

- [Python Data Science Handbook — Introduction to NumPy](https://jakevdp.github.io/PythonDataScienceHandbook/02.00-introduction-to-numpy.html) — free online; thorough coverage of arrays, broadcasting, masking, and ufuncs.
- [Scientific Python Lectures — NumPy](https://lectures.scientific-python.org/intro/numpy/index.html) — a concise, research-oriented tour of array creation, operations, and reductions.